# Amaliyot 10. Sinflar nomutanosibligi

10-interaktiv laboratoriyaga («Sinflar nomutanosibligi») hamroh. Kurs mavzusi: muvozanatsiz tanlanmalar, resampling va sinf vaznlari

Daftar oʻz-oʻzidan yetarli: birinchi katak `common.py` faylini oʻquv korpusi va yordamchi funksiyalar bilan yaratadi. Kataklarni yuqoridan pastga ishga tushiring. Korpus oʻquv (shablon) boʻlib, haqiqiy masalada oʻz CSV faylingizni quyidagi ustunlar bilan qoʻying: `text` va `label`.

In [ ]:
%%writefile common.py
"""Matnlarni tasniflash va klasterlash amaliyotlari uchun umumiy funksiyalar."""
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

TOPICS = {
    "sport": ("oʻyin jamoa gol oʻyinchi murabbiy chempionat turnir gʻalaba magʻlubiyat mavsum stadion muxlis hakam hisob "
              "himoyachi hujumchi darvozabon kubok liga final transfer klub zarba penalti musobaqa sportchi "
              "rekord medal olimpiada mashq durang").split(),
    "siyosat": ("qonun deputat saylov partiya ovoz prezident vazir hukumat parlament islohot byudjet "
                "sanksiya nomzod koalitsiya vazirlik farmon tuzatish loyiha muxolifat referendum diplomatiya "
                "hokim konstitutsiya majlis komissiya muzokara sammit saylovchi mandat vakolat").split(),
    "oshxona": ("shoʻrva retsept tuz goʻsht pishirmoq duxovka sabzavot tova qozon xamir shakar ziravor qaynatmoq qovurmoq "
                "dimlamoq sous bulyon salat oshpaz taom masalliq yogʻ un tuxum sut sarimsoq piyoz garmdori "
                "kartoshka shirinlik pishiriq marinad").split(),
    "texnologiya": ("protsessor dastur ilova smartfon neyrotarmoq algoritm maʼlumot server yangilanish dasturchi "
                    "kompyuter xotira ekran kamera brauzer tarmoq internet bulut kod platforma foydalanuvchi qurilma "
                    "chip batareya noutbuk dasturiy videokarta tizim interfeys shifrlash").split(),
}
NEUTRAL = ("bugun juda yangi aytishadi keyin ham mumkin edi boʻldi maʼlum qilindi kecha yaqinda hali hammasi "
           "bir nechta muhim birinchi").split()


def make_corpus(n_per_class=150, noise=0.3, length=(6, 11), seed=0):
    """4 mavzudagi qisqa «hujjatlar»dan iborat oʻquv korpusi.
    noise: oʻz mavzusidan boʻlmagan soʻzlar ulushi (neytral yoki begona). noise qancha katta boʻlsa, masala shuncha qiyin."""
    rng = random.Random(seed)
    names = list(TOPICS)
    rows = []
    for label in names:
        others = [t for t in names if t != label]
        for _ in range(n_per_class):
            words = []
            for _ in range(rng.randint(*length)):
                u = rng.random()
                if u < noise * 0.5:
                    words.append(rng.choice(NEUTRAL))
                elif u < noise:
                    words.append(rng.choice(TOPICS[rng.choice(others)]))
                else:
                    words.append(rng.choice(TOPICS[label]))
            rows.append((" ".join(words), label))
    df = pd.DataFrame(rows, columns=["text", "label"])
    return df.sample(frac=1, random_state=seed).reset_index(drop=True)


def project_2d(X, seed=0):
    """TF-IDF siyrak matritsasini tekislikka proyeksiyalash (TruncatedSVD)."""
    from sklearn.decomposition import TruncatedSVD
    return TruncatedSVD(2, random_state=seed).fit_transform(X)


def scatter2d(Z, labels, title="", ax=None):
    ax = ax or plt.gca()
    codes, uniq = pd.factorize(pd.Series(labels))
    sc = ax.scatter(Z[:, 0], Z[:, 1], c=codes, cmap="tab10", s=14, alpha=0.8)
    ax.set_title(title)
    handles = [plt.Line2D([], [], marker="o", ls="", color=sc.cmap(sc.norm(i)), label=str(u)) for i, u in enumerate(uniq)]
    ax.legend(handles=handles, fontsize=8)
    return ax


def top_terms(coef, terms, class_names, n=8):
    """Har bir sinf uchun eng katta vaznli soʻzlarni chiqaradi (chiziqli modellar)."""
    terms = np.asarray(terms)
    for i, name in enumerate(class_names):
        idx = np.argsort(coef[i])[::-1][:n]
        print(f"{name:12s}", ", ".join(terms[idx]))


In [ ]:
from common import *
big = make_corpus(n_per_class=300, noise=0.55, seed=1)
rare = "siyosat"
imb = pd.concat([big[big.label != rare], big[big.label == rare].sample(15, random_state=0)]).sample(frac=1, random_state=0)
imb["y"] = (imb.label == rare).astype(int)
print(imb.y.value_counts().to_dict(), "kam uchraydigan sinf ulushi:", round(imb.y.mean(), 3))

## 1. Bazaviy model va aniqlik paradoksi

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.utils import resample

Xtr_t, Xte_t, ytr, yte = train_test_split(imb.text, imb.y, test_size=0.3, stratify=imb.y, random_state=0)
vec = TfidfVectorizer().fit(Xtr_t)
Xtr, Xte = vec.transform(Xtr_t), vec.transform(Xte_t)
def report(name, model, Xt=Xtr, yt=ytr):
    model.fit(Xt, yt); p = model.predict(Xte)
    return dict(model=name, accuracy=accuracy_score(yte, p), precision=precision_score(yte, p, zero_division=0),
                recall=recall_score(yte, p), F1=f1_score(yte, p))
print("Doim «kam uchramaydigan»: accuracy =", round(1 - yte.mean(), 3))
rows = [report("ishlovsiz", LogisticRegression(max_iter=1000))]
rows.append(report("sinf vaznlari", LogisticRegression(max_iter=1000, class_weight="balanced")))
pd.DataFrame(rows).round(3)

## 2. Resampling (faqat oʻquv tanlanmasida!)

In [ ]:
import scipy.sparse as sp
tr = pd.DataFrame({"t": ytr.index, "y": ytr.values})
pos, neg = tr[tr.y == 1], tr[tr.y == 0]
over = pd.concat([neg, resample(pos, n_samples=len(neg), random_state=0)])
under = pd.concat([resample(neg, n_samples=len(pos), replace=False, random_state=0), pos])
def fit_subset(sub, name):
    Xs = vec.transform(imb.loc[sub.t, "text"]); return report(name, LogisticRegression(max_iter=1000), Xs, sub.y.values)
rows += [fit_subset(over, "oversampling"), fit_subset(under, "undersampling")]
try:
    from imblearn.over_sampling import SMOTE
    Xs, ys = SMOTE(random_state=0, k_neighbors=3).fit_resample(Xtr, ytr)
    rows.append(report("SMOTE", LogisticRegression(max_iter=1000), Xs, ys))
except ImportError:
    print("SMOTE uchun: !pip install imbalanced-learn")
pd.DataFrame(rows).round(3)

## 3. Qaror chegarasi va PR-egri chizigʻi

In [ ]:
from sklearn.metrics import precision_recall_curve
m = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
pr = m.predict_proba(Xte)[:, 1]
P, R, T = precision_recall_curve(yte, pr)
f1 = 2 * P * R / (P + R + 1e-9)
best = T[np.argmax(f1[:-1])]
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(R, P); ax[0].set_xlabel("recall"); ax[0].set_ylabel("precision"); ax[0].set_title("PR-egri chizigʻi")
ax[1].plot(T, P[:-1], label="precision"); ax[1].plot(T, R[:-1], label="recall"); ax[1].axvline(best, color="gray", ls="--", label=f"eng yaxshi chegara {best:.2f}")
ax[1].set_xlabel("chegara"); ax[1].legend(); plt.show()
print(classification_report(yte, (pr >= best).astype(int), zero_division=0))

## Topshiriqlar

1. Kam uchraydigan sinf ulushini 20%, 5%, 1% da chiqaring. Turli usullar qanday oʻzini tutadi?
2. Nima uchun resamplingni train va test ga boʻlishdan oldin qilib boʻlmaydi?
3. Chegarani «recall 0.9 dan kam boʻlmasin» talabiga koʻra tanlang. Qanday precision hosil boʻladi?